# 10. Feature Engineering (피처 엔지니어링)

## 목적
전처리된 데이터에서 ML 학습용 파생 피처 생성

## 입력
- `preprocessed.csv`: 전처리된 데이터

## 출력
- `features_final.csv`: 최종 피처셋

## 피처 엔지니어링 범위
- [x] Rolling 통계량 (mean, std, min, max)
- [x] 추세 피처 (delta, slope)
- [x] 파생 피처 (Shock Index, MEWS 등)
- [x] 인구통계학적 인코딩

In [1]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

print("=== 10. Feature Engineering 시작 ===")

=== 10. Feature Engineering 시작 ===


## Step 1: 데이터 로드

In [2]:
# --- 1-1: 데이터 로드 ---
print("\nStep 1: 데이터 로드")

df = pd.read_csv(os.path.join(INPUT_DIR, 'preprocessed.csv'))
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)

print(f"✓ 데이터 로드 완료: {len(df):,} rows")

# --- 1-2: 피처 그룹 정의 ---
# 메인 피처 (전처리 완료, 결측 없음)
vital_features = ['hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp']
lab_features = ['lactate', 'creatinine', 'wbc', 'platelets', 'potassium', 'sodium']
gcs_features = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_features = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']

# Missing Flag (전처리에서 생성)
missing_flags = ['lactate_missing', 'gcs_missing_flag', 'urine_missing_flag']

# Original (분석용, 모델 제외)
original_features = [
    'gcs_eye_original', 'gcs_verbal_original', 'gcs_motor_original', 'gcs_total_original',
    'urine_ml_6h_original', 'urine_ml_kg_hr_avg_original', 'oliguria_flag_original'
]

print(f"\n피처 그룹:")
print(f"  - Vital: {len(vital_features)}개")
print(f"  - Lab: {len(lab_features)}개")
print(f"  - GCS: {len(gcs_features)}개")
print(f"  - Urine: {len(urine_features)}개")
print(f"  - Missing Flags: {len(missing_flags)}개")
print(f"  - Original (모델 제외): {len(original_features)}개")


Step 1: 데이터 로드
✓ 데이터 로드 완료: 941,817 rows

피처 그룹:
  - Vital: 7개
  - Lab: 6개
  - GCS: 4개
  - Urine: 3개
  - Missing Flags: 3개
  - Original (모델 제외): 7개


## Step 2: Rolling 통계량

In [3]:
# ==============================================================================
# Step 2: Rolling 통계량 (과거 6개 윈도우 기준)
# ==============================================================================
# 
# 목적: 윈도우 간 변화 패턴 포착
# - mean: 평균 수준
# - std: 변동성 (불안정 신호)
# - min: 최저점 (저혈압/저산소 감지)
# - max: 최고점 (빈맥/고혈압 감지)
#
# 주의: 10_merge에서 이미 윈도우 내 집계했음
#       여기서는 "윈도우 간" 추세를 보는 것 (다른 레벨)
# ==============================================================================

print("\nStep 2: Rolling 통계량 (과거 6개 윈도우)")

rolling_features = ['hr', 'sbp', 'mbp', 'spo2', 'rr']
rolling_stats = ['mean', 'std', 'min', 'max']

for feat in rolling_features:
    if feat not in df.columns:
        print(f"  ⚠️ {feat} 컬럼 없음, 스킵")
        continue
    
    grouped = df.groupby('stay_id')[feat]
    
    df[f'{feat}_mean_6h'] = grouped.transform(lambda x: x.rolling(6, min_periods=1).mean())
    df[f'{feat}_std_6h'] = grouped.transform(lambda x: x.rolling(6, min_periods=1).std())
    df[f'{feat}_min_6h'] = grouped.transform(lambda x: x.rolling(6, min_periods=1).min())
    df[f'{feat}_max_6h'] = grouped.transform(lambda x: x.rolling(6, min_periods=1).max())

# std NaN 처리 (첫 윈도우는 std 계산 불가)
std_cols = [col for col in df.columns if '_std_6h' in col]
df[std_cols] = df[std_cols].fillna(0)

print(f"✓ Rolling 통계량 추가: {len(rolling_features)} × {len(rolling_stats)} = {len(rolling_features) * len(rolling_stats)}개 피처")


Step 2: Rolling 통계량 (과거 6개 윈도우)
✓ Rolling 통계량 추가: 5 × 4 = 20개 피처


## Step 3: 추세 피처 (Delta, Slope)

In [4]:
# ==============================================================================
# Step 3: 추세 피처 (Delta, Slope)
# ==============================================================================
#
# 목적: 시간에 따른 변화 방향과 속도 포착
# - delta_1h: 1시간 전 대비 변화량 (즉각적 변화)
# - delta_3h: 3시간 전 대비 변화량 (단기 추세)
# - slope_3h: 3시간 기울기 (변화 속도)
#
# 조기 악화 예측에서 "추세"가 핵심
# 예: HR 130 자체보다 "HR이 75→130으로 급상승" 정보가 더 중요
# ==============================================================================

print("\nStep 3: 추세 피처 (Delta, Slope)")

trend_features = ['hr', 'sbp', 'mbp', 'spo2', 'lactate']

for feat in trend_features:
    if feat not in df.columns:
        print(f"  ⚠️ {feat} 컬럼 없음, 스킵")
        continue
    
    grouped = df.groupby('stay_id')[feat]
    
    # Delta (차분)
    df[f'{feat}_delta_1h'] = grouped.diff(1)
    df[f'{feat}_delta_3h'] = grouped.diff(3)
    
    # Slope (기울기)
    def calc_slope(x):
        if len(x) < 2:
            return 0
        try:
            return np.polyfit(range(len(x)), x, 1)[0]
        except:
            return 0
    
    df[f'{feat}_slope_3h'] = grouped.transform(
        lambda x: x.rolling(3, min_periods=2).apply(calc_slope, raw=True)
    )

# NaN 처리 (첫 몇 윈도우는 계산 불가)
delta_slope_cols = [col for col in df.columns if '_delta_' in col or '_slope_' in col]
df[delta_slope_cols] = df[delta_slope_cols].fillna(0)

print(f"✓ 추세 피처 추가: {len(trend_features)} × 3 = {len(trend_features) * 3}개 피처")


Step 3: 추세 피처 (Delta, Slope)
✓ 추세 피처 추가: 5 × 3 = 15개 피처


## Step 4: 파생 피처

In [5]:
# ==============================================================================
# Step 4: 파생 피처
# ==============================================================================
#
# 임상적으로 의미 있는 복합 지표
# - Shock Index (SI): HR / SBP → 정상 0.5~0.7, >1.0 쇼크 의심
# - Modified SI: HR / MAP → 유사 목적
# - Pulse Pressure: SBP - DBP → 심박출량 간접 지표
# ==============================================================================

print("\nStep 4: 파생 피처")

# --- Shock Index = HR / SBP ---
# 정상: 0.5~0.7, 쇼크: >1.0
df['shock_index'] = df['hr'] / df['sbp'].replace(0, np.nan)
df['shock_index'] = df['shock_index'].clip(0, 3).fillna(1)

# --- Modified Shock Index = HR / MAP ---
df['modified_shock_index'] = df['hr'] / df['mbp'].replace(0, np.nan)
df['modified_shock_index'] = df['modified_shock_index'].clip(0, 5).fillna(1)

# --- Pulse Pressure = SBP - DBP ---
# 정상: 40~60 mmHg
df['pulse_pressure'] = df['sbp'] - df['dbp']
df['pulse_pressure'] = df['pulse_pressure'].clip(0, 150)

print("✓ 파생 피처 추가:")
print(f"  - shock_index: mean={df['shock_index'].mean():.2f}")
print(f"  - modified_shock_index: mean={df['modified_shock_index'].mean():.2f}")
print(f"  - pulse_pressure: mean={df['pulse_pressure'].mean():.1f}")


Step 4: 파생 피처
✓ 파생 피처 추가:
  - shock_index: mean=0.69
  - modified_shock_index: mean=1.03
  - pulse_pressure: mean=55.6


## Step 5: 중증도 점수 (MEWS, NEWS)

In [6]:
# ==============================================================================
# Step 5: 중증도 점수 (MEWS, NEWS) - 벡터화 버전
# ==============================================================================
#
# MEWS (Modified Early Warning Score): HR, SBP, RR, Temp 기반
# NEWS (National Early Warning Score): SpO2, SBP, HR 기반
#
# 벡터화: apply(axis=1) 대신 np.where 사용 → 5~15분 → 2~5초
# ==============================================================================

print("\nStep 5: 중증도 점수 계산 (벡터화)")

# --- MEWS 벡터화 ---
def calc_mews_vectorized(df):
    """
    Modified Early Warning Score (벡터화)
    - HR: 40미만/130초과(3), 50미만/110초과(2), 60미만/100초과(1)
    - SBP: 70미만(3), 80미만(2), 100미만(1)
    - RR: 9미만/30초과(3), 25초과(2), 12미만/20초과(1)
    - Temp: 35미만/39초과(2), 36미만/38초과(1)
    """
    score = pd.Series(0, index=df.index)
    
    # HR
    hr = df['hr']
    score += np.where((hr < 40) | (hr > 130), 3,
             np.where((hr < 50) | (hr > 110), 2,
             np.where((hr < 60) | (hr > 100), 1, 0)))
    
    # SBP
    sbp = df['sbp']
    score += np.where(sbp < 70, 3,
             np.where(sbp < 80, 2,
             np.where(sbp < 100, 1, 0)))
    
    # RR
    rr = df['rr']
    score += np.where((rr < 9) | (rr > 30), 3,
             np.where(rr > 25, 2,
             np.where((rr < 12) | (rr > 20), 1, 0)))
    
    # Temp
    temp = df['temp']
    score += np.where((temp < 35) | (temp > 39), 2,
             np.where((temp < 36) | (temp > 38), 1, 0))
    
    return score

# --- NEWS 벡터화 ---
def calc_news_vectorized(df):
    """
    National Early Warning Score - Simplified (벡터화)
    - SpO2: 91이하(3), 93이하(2), 95이하(1)
    - SBP: 90이하/220이상(3), 100이하(2), 110이하(1)
    - HR: 40이하/131이상(3), 111이상(2), 50이하/91이상(1)
    """
    score = pd.Series(0, index=df.index)
    
    # SpO2
    spo2 = df['spo2']
    score += np.where(spo2 <= 91, 3,
             np.where(spo2 <= 93, 2,
             np.where(spo2 <= 95, 1, 0)))
    
    # SBP
    sbp = df['sbp']
    score += np.where((sbp <= 90) | (sbp >= 220), 3,
             np.where(sbp <= 100, 2,
             np.where(sbp <= 110, 1, 0)))
    
    # HR
    hr = df['hr']
    score += np.where((hr <= 40) | (hr >= 131), 3,
             np.where(hr >= 111, 2,
             np.where((hr <= 50) | (hr >= 91), 1, 0)))
    
    return score

# 계산
df['mews_score'] = calc_mews_vectorized(df)
df['news_score'] = calc_news_vectorized(df)

print(f"✓ MEWS: mean={df['mews_score'].mean():.2f}, max={df['mews_score'].max()}")
print(f"✓ NEWS: mean={df['news_score'].mean():.2f}, max={df['news_score'].max()}")


Step 5: 중증도 점수 계산 (벡터화)
✓ MEWS: mean=0.97, max=9
✓ NEWS: mean=1.19, max=9


## Step 6: 인구통계학적 인코딩

In [7]:
# ==============================================================================
# Step 6: 인구통계학적 인코딩
# ==============================================================================
#
# 포함:
#   - gender_male: M=1, F=0
#   - anchor_age: 그대로 사용 (연속형, XGBoost에 적합)
#   - hours_in_icu: observation_hour 복사
#
# 제외:
#   - first_careunit (ICU type): 15개로 많고, 그룹핑 안 하기로 결정
# ==============================================================================

print("\nStep 6: 인구통계학적 인코딩")

# --- Gender 인코딩 ---
if 'gender' in df.columns:
    df['gender_male'] = (df['gender'] == 'M').astype(int)
    male_ratio = df['gender_male'].mean() * 100
    print(f"✓ gender_male: M=1, F=0 (남성 비율: {male_ratio:.1f}%)")

# --- Age 그대로 사용 ---
if 'anchor_age' in df.columns:
    print(f"✓ anchor_age: 그대로 사용 (mean={df['anchor_age'].mean():.1f})")

# --- Hours in ICU ---
df['hours_in_icu'] = df['observation_hour']
print(f"✓ hours_in_icu: observation_hour 복사")

# --- ICU type 제외 안내 ---
print(f"\n  ⓘ first_careunit (ICU type): 인코딩 제외 (15개 유형, 그룹핑 미적용)")


Step 6: 인구통계학적 인코딩
✓ gender_male: M=1, F=0 (남성 비율: 53.1%)
✓ anchor_age: 그대로 사용 (mean=64.1)
✓ hours_in_icu: observation_hour 복사

  ⓘ first_careunit (ICU type): 인코딩 제외 (15개 유형, 그룹핑 미적용)


## Step 7: 최종 피처 선택 및 저장

In [8]:
# ==============================================================================
# Step 7: 최종 피처 선택
# ==============================================================================
#
# 모델 입력 피처 구성:
#   - 메인 피처 (Vital, Lab, GCS, Urine)
#   - Rolling 통계량
#   - 추세 피처 (Delta, Slope)
#   - 파생 피처 (Shock Index 등)
#   - 중증도 점수 (MEWS, NEWS)
#   - Missing Flags
#   - 인구통계 (gender_male, anchor_age, hours_in_icu)
#
# 제외:
#   - *_original: 분석용 보존, 모델 제외 (결측 있음)
#   - gender, first_careunit: 원본 범주형
# ==============================================================================

print("\n" + "="*60)
print("Step 7: 최종 피처 선택")
print("="*60)

# --- ID 컬럼 ---
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']

# --- 레이블 컬럼 ---
label_cols = [col for col in df.columns if 'next_' in col]

# --- 제외 컬럼 ---
exclude_cols = (
    id_cols + 
    label_cols + 
    original_features +  # 분석용 보존, 모델 제외
    ['gender', 'first_careunit', 'age_group',  # 원본 범주형
     'deathtime', 'dnr_time', 'vent_start_time', 'pressor_start_time',  # 메타 정보
     'icu_mortality', 'hospital_mortality']  # 레이블 관련
)

# --- 피처 컬럼 (자동 수집) ---
feature_cols = [col for col in df.columns if col not in exclude_cols]

# --- 피처 그룹별 카운트 ---
rolling_cols = [col for col in feature_cols if '_6h' in col and col not in vital_features + lab_features + gcs_features + urine_features]
trend_cols = [col for col in feature_cols if '_delta_' in col or '_slope_' in col]
derived_cols = ['shock_index', 'modified_shock_index', 'pulse_pressure']
score_cols = ['mews_score', 'news_score']
demo_cols = ['gender_male', 'anchor_age', 'hours_in_icu']

print(f"\n=== 피처 구성 ===")
print(f"  Vital: {len([c for c in vital_features if c in feature_cols])}개")
print(f"  Lab: {len([c for c in lab_features if c in feature_cols])}개")
print(f"  GCS: {len([c for c in gcs_features if c in feature_cols])}개")
print(f"  Urine: {len([c for c in urine_features if c in feature_cols])}개")
print(f"  Rolling: {len([c for c in rolling_cols if c in feature_cols])}개")
print(f"  Trend: {len([c for c in trend_cols if c in feature_cols])}개")
print(f"  Derived: {len([c for c in derived_cols if c in feature_cols])}개")
print(f"  Scores: {len([c for c in score_cols if c in feature_cols])}개")
print(f"  Missing Flags: {len([c for c in missing_flags if c in feature_cols])}개")
print(f"  Demographics: {len([c for c in demo_cols if c in feature_cols])}개")
print(f"\n  총 피처 수: {len(feature_cols)}개")

# --- 최종 DataFrame ---
final_cols = [col for col in id_cols if col in df.columns] + feature_cols + label_cols
df_final = df[final_cols].copy()

print(f"\n=== 최종 DataFrame ===")
print(f"  행 수: {len(df_final):,}")
print(f"  컬럼 수: {len(df_final.columns)} (ID: {len([c for c in id_cols if c in df_final.columns])}, 피처: {len(feature_cols)}, 레이블: {len(label_cols)})")


Step 7: 최종 피처 선택

=== 피처 구성 ===
  Vital: 7개
  Lab: 6개
  GCS: 4개
  Urine: 3개
  Rolling: 20개
  Trend: 15개
  Derived: 3개
  Scores: 2개
  Missing Flags: 3개
  Demographics: 3개

  총 피처 수: 66개

=== 최종 DataFrame ===
  행 수: 941,817
  컬럼 수: 84 (ID: 6, 피처: 66, 레이블: 12)


## Step 8: 저장

In [11]:
print("\n" + "="*60)
print("Step 8: 저장")
print("="*60)

# --- 저장 ---
output_path = os.path.join(OUTPUT_DIR, 'features_final.csv')
df_final.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: features_final.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_final):,}개")
print(f"  - 컬럼 수: {len(df_final.columns)}개")
print(f"  - 경로: {output_path}")


Step 8: 저장

✓ 저장 완료: features_final.csv
  - 파일 크기: 752.08 MB
  - 행 수: 941,817개
  - 컬럼 수: 84개
  - 경로: ../data/processed/features_final.csv


In [12]:
# --- 피처 목록 출력 ---
print(f"\n=== 피처 목록 ({len(feature_cols)}개) ===")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

# --- 레이블 목록 출력 ---
print(f"\n=== 레이블 목록 ({len(label_cols)}개) ===")
for col in label_cols:
    print(f"  - {col}")


=== 피처 목록 (66개) ===
   1. anchor_age
   2. hr
   3. rr
   4. spo2
   5. temp
   6. sbp
   7. dbp
   8. mbp
   9. lactate
  10. creatinine
  11. wbc
  12. platelets
  13. potassium
  14. sodium
  15. gcs_eye
  16. gcs_verbal
  17. gcs_motor
  18. gcs_total
  19. urine_ml_6h
  20. urine_ml_kg_hr_avg
  21. oliguria_flag
  22. lactate_missing
  23. gcs_missing_flag
  24. urine_missing_flag
  25. hr_mean_6h
  26. hr_std_6h
  27. hr_min_6h
  28. hr_max_6h
  29. sbp_mean_6h
  30. sbp_std_6h
  31. sbp_min_6h
  32. sbp_max_6h
  33. mbp_mean_6h
  34. mbp_std_6h
  35. mbp_min_6h
  36. mbp_max_6h
  37. spo2_mean_6h
  38. spo2_std_6h
  39. spo2_min_6h
  40. spo2_max_6h
  41. rr_mean_6h
  42. rr_std_6h
  43. rr_min_6h
  44. rr_max_6h
  45. hr_delta_1h
  46. hr_delta_3h
  47. hr_slope_3h
  48. sbp_delta_1h
  49. sbp_delta_3h
  50. sbp_slope_3h
  51. mbp_delta_1h
  52. mbp_delta_3h
  53. mbp_slope_3h
  54. spo2_delta_1h
  55. spo2_delta_3h
  56. spo2_slope_3h
  57. lactate_delta_1h
  58. lactate_delt

In [13]:
print("\n=== 10. Feature Engineering 완료 ===")


=== 10. Feature Engineering 완료 ===


---

In [14]:
# 결측 확인
print("=== 결측 확인 ===")
missing = df_final[feature_cols].isna().sum()
if missing.sum() == 0:
    print("✓ 결측 없음")
else:
    print(missing[missing > 0])

# 상관관계 높은 피처 확인 (다중공선성)
print("\n=== 고상관 피처 쌍 (>0.95) ===")
corr = df_final[feature_cols].corr()
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.95:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
for pair in high_corr[:10]:
    print(f"  {pair[0]} vs {pair[1]}: {pair[2]:.3f}")

=== 결측 확인 ===
✓ 결측 없음

=== 고상관 피처 쌍 (>0.95) ===
  hr vs hr_mean_6h: 0.977
  hr vs hr_min_6h: 0.969
  hr vs hr_max_6h: 0.967
  rr vs rr_mean_6h: 0.952
  sbp vs sbp_mean_6h: 0.961
  mbp vs mbp_mean_6h: 0.954
  hr_mean_6h vs hr_min_6h: 0.988
  hr_mean_6h vs hr_max_6h: 0.989
  hr_min_6h vs hr_max_6h: 0.958
  sbp_mean_6h vs sbp_min_6h: 0.982
